In [ ]:
# Post-training metrics: Test NLL, ECE, OOD (CIFAR-10 vs STL-10, CIFAR-100 vs SVHN), Predictive Entropy vs ECDF, 
# During training: Stepsizes, mean ESS, accuracy/losses (?)

In [ ]:
import os
import glob
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import (
    accuracy_score, log_loss,
    confusion_matrix, roc_auc_score
)
from sklearn.calibration import calibration_curve, CalibrationDisplay
from sklearn.manifold import MDS

# 1) Config & helpers
DATA_DIR = './data'
WEIGHTS_DIR = './sgld_weights'  # adjust to your path
BATCH_SIZE = 256
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# 2) Load test sets
test_c10 = DataLoader(
    datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=transform),
    batch_size=BATCH_SIZE, shuffle=False)
test_c100 = DataLoader(
    datasets.CIFAR100(DATA_DIR, train=False, download=True, transform=transform),
    batch_size=BATCH_SIZE, shuffle=False)
ood_stl10 = DataLoader(
    datasets.STL10(DATA_DIR, split='test', download=True, transform=transform),
    batch_size=BATCH_SIZE, shuffle=False)
ood_svhn = DataLoader(
    datasets.SVHN(DATA_DIR, split='test', download=True, transform=transform),
    batch_size=BATCH_SIZE, shuffle=False)

# 3) Model builder (must match your training arch)
import models.resnet as resnet_py  # assumes models/resnet.py exists
def make_model():
    model = resnet_py.ResNet18(num_classes=10)
    return model.to(DEVICE).eval()

# 4) Load all weight samples
weight_files = sorted(glob.glob(os.path.join(WEIGHTS_DIR, '*.pt')))
models = []
for wf in weight_files:
    m = make_model()
    m.load_state_dict(torch.load(wf, map_location=DEVICE))
    models.append(m)

# 5) Get predictive probs & entropies
import torch.nn.functional as F

def predict_list(dataloader, num_classes):
    all_probs = []
    all_labels = []
    for x, y in dataloader:
        x = x.to(DEVICE)
        # Stack predictions from each weight sample
        ps = torch.stack([F.softmax(m(x), dim=1) for m in models], dim=0)  # SxBxC
        mean_p = ps.mean(0)  # BxC
        # predictive entropy H[ŷ]
        entropy = - (mean_p * mean_p.log()).sum(1).cpu().numpy()
        # expected entropy E[H[y|w]]
        exp_ent = - (ps * ps.log()).sum(2).mean(0).cpu().numpy()
        # mutual information MI = H[ŷ] - E[H[y|w]]
        mi = entropy - exp_ent
        all_probs.append(mean_p.cpu().numpy())
        all_labels.append(y.numpy())
        if 'entropy_list' not in globals():
            entropy_list = entropy
            mi_list = mi
        else:
            entropy_list = np.concatenate([entropy_list, entropy])
            mi_list = np.concatenate([mi_list, mi])
    return np.vstack(all_probs), np.concatenate(all_labels), entropy_list, mi_list

# 6) Evaluate on CIFAR-10
probs_c10, y_c10, ent_c10, mi_c10 = predict_list(test_c10, num_classes=10)

# Test acc, NLL
y_pred = np.argmax(probs_c10, axis=1)
acc = accuracy_score(y_c10, y_pred)
nll = log_loss(y_c10, probs_c10)
print(f"CIFAR-10  Acc={acc:.4f}, NLL={nll:.4f}")

# 7) Confusion‐style MI matrix
cm = confusion_matrix(y_c10, y_pred, labels=range(10))
mi_mat = np.zeros((10,10))
counts = np.zeros((10,10))
for i in range(len(y_c10)):
    t, p = y_c10[i], y_pred[i]
    mi_mat[t,p] += mi_c10[i]
    counts[t,p] += 1
mi_mat /= np.maximum(counts, 1)
plt.imshow(mi_mat, cmap='viridis')
plt.colorbar(); plt.title("Avg MI per (true,pred)"); plt.show()

# 8) Predictive entropy vs ECDF
sorted_e = np.sort(ent_c10)
ecdf = np.arange(1, len(sorted_e)+1) / len(sorted_e)
plt.plot(sorted_e, ecdf); plt.xlabel("Predictive Entropy"); plt.ylabel("ECDF")
plt.title("Entropy ECDF"); plt.show()

# 9) MDS in weight space
# Flatten each weight sample into a vector
w_vectors = []
for wf in weight_files:
    state = torch.load(wf, map_location='cpu')
    w_vectors.append(np.concatenate([v.flatten().numpy() for v in state.values()]))
w_vectors = np.vstack(w_vectors)
mds = MDS(n_components=2, random_state=0).fit_transform(w_vectors)
plt.scatter(mds[:,0], mds[:,1]); plt.title("MDS of Weight Samples"); plt.show()

# 10) Interpolation loss curve
w0 = w_vectors[0]; w1 = w_vectors[-1]
alphas = np.linspace(0,1,21)
loss_interp = []
for α in alphas:
    # load mixed weights into model
    w_mix = (1-α)*w0 + α*w1
    # assign back to model[0]
    state = models[0].state_dict()
    idx = 0
    for k in state:
        sz = state[k].numel()
        state[k] = torch.tensor(w_mix[idx:idx+sz].reshape(state[k].shape))
        idx += sz
    models[0].load_state_dict(state)
    # compute NLL on test
    probs, y, _, _ = predict_list(test_c10, num_classes=10)
    loss_interp.append(log_loss(y, probs))
plt.plot(alphas, loss_interp); plt.xlabel("α"); plt.ylabel("NLL"); plt.title("Interpolation Loss"); plt.show()

# 11) Reliability diagram
prob_pos = probs_c10[np.arange(len(y_c10)), y_pred]
disp = CalibrationDisplay.from_predictions(y_c10==y_pred, prob_pos, n_bins=10)
disp.plot(); plt.title("Reliability Diagram"); plt.show()

# 12) OOD Detection AUCROC
# here we use predictive entropy as score
def ood_auc(in_ent, out_ent):
    y_true = np.concatenate([np.zeros_like(in_ent), np.ones_like(out_ent)])
    scores = np.concatenate([in_ent, out_ent])
    return roc_auc_score(y_true, scores)

auc_c10_stl = ood_auc(ent_c10, predict_list(ood_stl10, 10)[2])
auc_c100_svhn = ood_auc(predict_list(test_c100, 100)[2], predict_list(ood_svhn, 10)[2])
print("AUC CIFAR-10 vs STL-10:", auc_c10_stl)
print("AUC CIFAR-100 vs SVHN:", auc_c100_svhn)
